In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

results_dir = project_root / 'results'


In [ ]:
df = pd.read_csv(results_dir / 'mnist_dp_batchsize_tradeoff.csv')
df.head()


In [ ]:
summary_df = df.groupby(['batch_size', 'dp_sigma']).agg({
    'attack_ssim': ['mean', 'std'],
    'test_acc': ['mean', 'std']
}).reset_index()

summary_df.columns = [
    '_'.join(col).strip() if col[1] else col[0]
    for col in summary_df.columns.values
]
summary_df = summary_df.rename(columns={'batch_size_': 'batch_size', 'dp_sigma_': 'dp_sigma'})

summary_df['privacy_score'] = 1 - summary_df['attack_ssim_mean']
summary_df['tradeoff_score'] = 0.5 * summary_df['privacy_score'] + 0.5 * summary_df['test_acc_mean']
summary_df.head()


In [ ]:
ranking_df = summary_df.sort_values('tradeoff_score', ascending=False).reset_index(drop=True)
ranking_path = results_dir / 'mnist_tradeoff_ranking.csv'
ranking_df.to_csv(ranking_path, index=False)

print('Top configurations by trade-off score:')
display(ranking_df[[
    'batch_size',
    'dp_sigma',
    'attack_ssim_mean',
    'test_acc_mean',
    'privacy_score',
    'tradeoff_score'
]].head(10))


In [ ]:
ssim_pivot = summary_df.pivot(index='batch_size', columns='dp_sigma', values='attack_ssim_mean')
acc_pivot = summary_df.pivot(index='batch_size', columns='dp_sigma', values='test_acc_mean')
tradeoff_pivot = summary_df.pivot(index='batch_size', columns='dp_sigma', values='tradeoff_score')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(ssim_pivot, annot=True, fmt='.3f', cmap='viridis', ax=axes[0])
axes[0].set_title('Mean Attack SSIM')

sns.heatmap(acc_pivot, annot=True, fmt='.3f', cmap='viridis', ax=axes[1])
axes[1].set_title('Mean Test Accuracy')

sns.heatmap(tradeoff_pivot, annot=True, fmt='.3f', cmap='viridis', ax=axes[2])
axes[2].set_title('Privacy-Utility Trade-off Score')

for ax in axes:
    ax.set_xlabel('DP Sigma')
    ax.set_ylabel('Batch Size')

plt.tight_layout()


In [ ]:
plot_path = results_dir / 'mnist_tradeoff_heatmaps.png'
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()

print(f'Saved ranking to: {ranking_path}')
print(f'Saved figure to: {plot_path}')
